<a href="https://colab.research.google.com/github/Sharmila-lab/Heart-Rate-Prediction-Model/blob/main/Untitle_flipcartb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
!pip install transformers datasets torch scikit-learn accelerate

In [28]:
from google.colab import files
uploaded = files.upload()

Saving Flipkart_reviews.csv to Flipkart_reviews.csv


In [29]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

In [30]:
# To resolve FileNotFoundError, ensure 'Flipkart_reviews.csv' is in your Colab environment.
# You can upload it via the files panel, or download it using a command like:
# !wget -O /content/Flipkart_reviews.csv "YOUR_DOWNLOAD_LINK_HERE"

df = pd.read_csv("/content/Flipkart_reviews - Copy.csv") # Corrected filename
print(df.head())
print("dataset size:", len(df))

                                        product_name product_price Rate  \
0  Candes 12 L Room/Personal Air Cooler??????(Whi...          3999    5   
1  Candes 12 L Room/Personal Air Cooler??????(Whi...          3999    5   
2  Candes 12 L Room/Personal Air Cooler??????(Whi...          3999    3   
3  Candes 12 L Room/Personal Air Cooler??????(Whi...          3999    1   
4  Candes 12 L Room/Personal Air Cooler??????(Whi...          3999    3   

            Review                                            Summary  \
0           super!  great cooler excellent air flow and for this p...   
1          awesome              best budget 2 fit cooler nice cooling   
2             fair  the quality is good but the power of air is de...   
3  useless product                  very bad product its a only a fan   
4             fair                                      ok ok product   

  Sentiment  
0  positive  
1  positive  
2  positive  
3  negative  
4   neutral  
dataset size: 49999


In [31]:
df["Review"]=df["Review"].fillna("")
df["Summary"]=df["Summary"].fillna("")
df["text"]=df["Review"]+" "+df["Summary"]

#remove empty rows
df=df[df["text"].str.strip() !=""]
print("After cleaning:", len(df))

After cleaning: 49998


In [32]:
label_encoder=LabelEncoder()
df["label"]=label_encoder.fit_transform(df["Sentiment"])
print("Classes:",label_encoder.classes_)

Classes: ['negative' 'neutral' 'positive']


In [33]:
train_df, test_df=train_test_split(
    df[["text","label"]],
    test_size=0.2,
    random_state=42
)

In [34]:
#convert to huggingface dataset
train_dataset=Dataset.from_pandas(train_df)
test_dataset=Dataset.from_pandas(test_df)
print(train_dataset.column_names)

['text', 'label', '__index_level_0__']


In [35]:
#convert to huggingface dataset
train_dataset=Dataset.from_pandas(train_df)
test_dataset=Dataset.from_pandas(test_df)
print(train_dataset.column_names)

['text', 'label', '__index_level_0__']


In [36]:
#load tokenizer
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")

In [37]:
#tokenization function
def tokenize_function(example):
  texts=[str(t) for t in example["text"]]
  return tokenizer(texts, padding="max_length", truncation=True, max_length=128)

In [38]:
train_dataset=train_dataset.map(tokenize_function, batched=True)
test_dataset=test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/39998 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [39]:

#remove unnecessary columns
train_dataset=train_dataset.remove_columns(["text","__index_level_0__"])
test_dataset=test_dataset.remove_columns(["text","__index_level_0__"])


In [40]:
#set format for PyTorch
train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [41]:
#load BERT model
model=BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [42]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

In [43]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [44]:
#evaluate model
results=trainer.evaluate()

In [45]:
#detailed classification report
predictions=trainer.predict(test_dataset)
preds=predictions.predictions.argmax(-1)
print(classification_report(test_dataset["label"], preds))

              precision    recall  f1-score   support

           0       0.09      0.38      0.15      1418
           1       0.04      0.22      0.07       446
           2       0.77      0.16      0.26      8136

    accuracy                           0.19     10000
   macro avg       0.30      0.25      0.16     10000
weighted avg       0.64      0.19      0.24     10000



In [46]:
def predict_sentiment(text):
  device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  inputs=tokenizer(
      text,
      return_tensors="pt",
      truncation=True,
      padding=True
  )
  inputs={key:value.to(device)for key, value in inputs.items()}
  with torch.no_grad():
    outputs=model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([prediction])[0]


In [47]:
print(predict_sentiment("very bad cooler"))
print(predict_sentiment("excellent airflow and amazing cooling"))

negative
neutral
